# Катание на CartPole

**ML8 · Reinforcement Learning** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~20 min

## Введение

Проблема, которую мы решали на предыдущем уроке, может показаться игрушечной, не совсем применимой к реальным ситуациям. Это не так, потому что многие реальные задачи имеют похожий сценарий – включая игру в шахматы или го. Они похожи тем, что у нас есть доска с установленными правилами и **дискретное состояние**.

В этом уроке мы применим те же принципы Q-обучения к задаче с **непрерывным состоянием**, то есть состоянием, заданным одним или несколькими действительными числами. Мы рассмотрим следующую задачу:

> **Задача**: Если Пётр хочет убежать от волка, ему нужно уметь двигаться быстрее. Мы увидим, как Пётр может научиться кататься на коньках, в частности, сохранять равновесие, используя Q-обучение.

![Великое бегство!](ms-assets/ml/escape.18862db9930337e3.webp)

> Пётр и его друзья проявляют изобретательность, чтобы убежать от волка! Изображение от [Jen Looper](https://twitter.com/jenlooper)

Мы будем использовать упрощённую версию балансирования, известную как задача **CartPole**. В мире cartpole у нас есть горизонтальный ползунок, который может двигаться влево или вправо, а цель — удержать вертикальный стержень на ползунке в равновесии.

![a cartpole](ms-assets/ml/cartpole.b5609cc0494a14f7.webp)



## Предварительные знания

В этом уроке мы будем использовать библиотеку **OpenAI Gym** для симуляции различных **окружений**. Код этого урока можно запускать локально (например, из Visual Studio Code), в этом случае симуляция откроется в новом окне. При запуске кода онлайн может потребоваться внести некоторые изменения в код, как описано [здесь](https://towardsdatascience.com/rendering-openai-gym-envs-on-binder-and-google-colab-536f99391cc7).



## OpenAI Gym

На прошлом уроке правила игры и состояние задавались классом `Board`, который мы определили сами. Здесь мы будем использовать специальное **симуляционное окружение**, которое будет моделировать физику балансирующего стержня. Одно из самых популярных симуляционных окружений для обучения алгоритмов обучения с подкреплением называется [Gym](https://gymnasium.farama.org/), поддерживаемое [OpenAI](https://openai.com/). Используя этот gym, мы можем создавать разные **окружения** — от симуляции cartpole до игр Atari.

> **Примечание**: Другие доступные окружения OpenAI Gym можно посмотреть [здесь](https://gymnasium.farama.org/environments/classic_control/).

Сначала установим gym и импортируем необходимые библиотеки (блок кода 1):



In [ ]:
import sys
%pip install "gym==0.25.2" 

import gym
import matplotlib.pyplot as plt
import numpy as np
import random

## Упражнение - инициализировать окружение cartpole

Чтобы работать с задачей балансирования cartpole, нам нужно инициализировать соответствующее окружение. Каждое окружение связано с:

- **Пространством наблюдений (Observation space)**, которое определяет структуру информации, которую мы получаем из окружения. Для задачи cartpole мы получаем позицию стержня, скорость и некоторые другие значения.

- **Пространством действий (Action space)**, которое определяет возможные действия. В нашем случае пространство действий дискретное и состоит из двух действий — **влево** и **вправо**. (блок кода 2)

1. Для инициализации введите следующий код:

    ```python
    env = gym.make("CartPole-v1")
    print(env.action_space)
    print(env.observation_space)
    print(env.action_space.sample())
    ```

Чтобы понять, как работает окружение, давайте запустим короткую симуляцию на 100 шагов. На каждом шаге мы будем задавать одно из возможных действий — в этой симуляции мы просто случайным образом выбираем действие из `action_space`.

1. Запустите код ниже и посмотрите, к чему это приведёт.

    ✅ Помните, что предпочтительно выполнять этот код на локальной установке Python! (блок кода 3)

    ```python
    env.reset()
    
    for i in range(100):
       env.render()
       env.step(env.action_space.sample())
    env.close()
    ```

    Вы должны увидеть нечто подобное на это изображение:

    ![cartpole без балансировки](ms-assets/ml/8-Reinforcement/2-Gym/images/cartpole-nobalance.gif)

1. Во время симуляции нам нужно получать наблюдения, чтобы решить, как действовать. По сути, функция step возвращает текущие наблюдения, функцию вознаграждения и флаг done, который показывает, есть ли смысл продолжать симуляцию: (блок кода 4)

    ```python
    env.reset()
    
    done = False
    while not done:
       env.render()
       obs, rew, done, info = env.step(env.action_space.sample())
       print(f"{obs} -> {rew}")
    env.close()
    ```

    В результате в выводе ноутбука вы увидите следующее:

    ```text
    [ 0.03403272 -0.24301182  0.02669811  0.2895829 ] -> 1.0
    [ 0.02917248 -0.04828055  0.03248977  0.00543839] -> 1.0
    [ 0.02820687  0.14636075  0.03259854 -0.27681916] -> 1.0
    [ 0.03113408  0.34100283  0.02706215 -0.55904489] -> 1.0
    [ 0.03795414  0.53573468  0.01588125 -0.84308041] -> 1.0
    ...
    [ 0.17299878  0.15868546 -0.20754175 -0.55975453] -> 1.0
    [ 0.17617249  0.35602306 -0.21873684 -0.90998894] -> 1.0
    ```

    Вектор наблюдений, который возвращается на каждом шаге симуляции, содержит следующие значения:
    - Позиция тележки
    - Скорость тележки
    - Угол стержня
    - Скорость вращения стержня

1. Получите минимальные и максимальные значения этих чисел: (блок кода 5)

    ```python
    print(env.observation_space.low)
    print(env.observation_space.high)
    ```

    Также вы можете заметить, что значение вознаграждения на каждом шаге симуляции всегда равно 1. Это потому, что наша цель — выживать как можно дольше, то есть удерживать стержень в достаточно вертикальном положении максимально долго.

    ✅ Фактически, симуляция CartPole считается решённой, если мы достигаем среднего вознаграждения 195 за 100 последовательных испытаний.



## Дискретизация состояния

В Q-обучении нам нужно построить Q-таблицу, которая определяет, что делать в каждом состоянии. Чтобы это сделать, наше состояние должно быть **дискретным**, точнее, должно содержать конечное число дискретных значений. Значит, нам нужно как-то **дискретизировать** наши наблюдения, отображая их в конечный набор состояний.

Есть несколько способов это сделать:

- **Разбиение на интервалы (bins)**. Если мы знаем интервал некоторого значения, мы можем разделить этот интервал на несколько **корзин (bins)** и заменить значение номером корзины, к которой оно принадлежит. Это можно сделать с помощью функции numpy [`digitize`](https://numpy.org/doc/stable/reference/generated/numpy.digitize.html). В этом случае мы точно знаем размерность состояния, так как она будет зависеть от количества выбранных корзин для дискретизации.
  
✅ Можно использовать линейную интерполяцию, чтобы привести значения к некоторому конечному интервалу (скажем, от -20 до 20), а затем преобразовать числа в целые, округляя их. Это даёт меньше контроля над размером состояния, особенно если вы не знаете точные диапазоны входных значений. Например, в нашем случае 2 из 4 значений не имеют верхних или нижних границ, что может привести к бесконечному количеству состояний.

В нашем примере мы выберем второй подход. Как вы увидите позже, несмотря на неопределённые границы, эти значения редко выходят за пределы некоторых конечных интервалов, значит состояния с экстремальными значениями будут очень редки.

1. Вот функция, которая принимает наблюдение из нашей модели и выдаёт кортеж из 4 целых чисел: (блок кода 6)

    ```python
    def discretize(x):
        return tuple((x/np.array([0.25, 0.25, 0.01, 0.1])).astype(np.int64))
    ```

1. Рассмотрим также другой метод дискретизации с помощью корзин: (блок кода 7)

    ```python
    def create_bins(i,num):
        return np.arange(num+1)*(i[1]-i[0])/num+i[0]
    
    print("Sample bins for interval (-5,5) with 10 bins\n",create_bins((-5,5),10))
    
    ints = [(-5,5),(-2,2),(-0.5,0.5),(-2,2)] # интервалы значений для каждого параметра
    nbins = [20,20,10,10] # количество корзин для каждого параметра
    bins = [create_bins(ints[i],nbins[i]) for i in range(4)]
    
    def discretize_bins(x):
        return tuple(np.digitize(x[i],bins[i]) for i in range(4))
    ```

1. Теперь запустим короткую симуляцию и посмотрим на эти дискретизированные значения окружения. Попробуйте оба варианта — `discretize` и `discretize_bins` — и посмотрите, есть ли разница.

    ✅ функция discretize_bins возвращает номер корзины, нумерация с 0. Значит, для значений входной переменной около 0 она возвращает число из середины интервала (10). В функции discretize мы не учитывали диапазон выходных значений, позволяя им быть отрицательными, поэтому значения состояний не сдвинуты, и 0 соответствует 0. (блок кода 8)

    ```python
    env.reset()
    
    done = False
    while not done:
       #env.render()
       obs, rew, done, info = env.step(env.action_space.sample())
       #print(discretize_bins(obs))
       print(discretize(obs))
    env.close()
    ```

    ✅ Раскомментируйте строку, начинающуюся с env.render, если хотите видеть выполнение окружения. В противном случае можно выполнять его в фоновом режиме — это быстрее. Мы будем использовать такое «невидимое» выполнение во время процесса Q-обучения.



## Структура Q-таблицы

В предыдущем уроке состояние было простой парой чисел от 0 до 8, и поэтому было удобно представлять Q-таблицу как numpy-тензор размером 8x8x4. Если мы используем дискретизацию корзинами, размерность нашего вектора состояния тоже известна, поэтому мы можем использовать тот же подход и представить состояние массивом формы 20x20x10x10x2 (где 2 — размерность пространства действий, а первые размеры соответствуют количеству корзин для каждого из параметров в пространстве наблюдений).

Однако иногда точный размер пространства наблюдений неизвестен. В случае функции `discretize` мы никогда не можем быть уверены, что наше состояние останется в определённых пределах, так как некоторые из исходных значений не ограничены. Поэтому мы используем немного другой подход и представим Q-таблицу словарём.

1. Используйте пару *(state,action)* как ключ словаря, а значением будет соответствующая запись Q-таблицы. (блок кода 9)

    ```python
    Q = {}
    actions = (0,1)
    
    def qvalues(state):
        return [Q.get((state,a),0) for a in actions]
    ```

    Здесь мы также определяем функцию `qvalues()`, которая возвращает список значений Q-таблицы для данного состояния, соответствующих всем возможным действиям. Если запись отсутствует в Q-таблице, возвращаем по умолчанию 0.



## Начинаем Q-обучение

Теперь мы готовы научить Петра балансировать!

1. Сначала зададим некоторые гиперпараметры: (блок кода 10)

    ```python
    # гиперпараметры
    alpha = 0.3
    gamma = 0.9
    epsilon = 0.90
    ```

    Здесь `alpha` — это **скорость обучения**, которая определяет, насколько сильно мы должны корректировать текущие значения Q-таблицы на каждом шаге. В предыдущем уроке мы начинали с 1, а затем уменьшали `alpha` во время обучения. В этом примере мы оставим её постоянной для простоты, хотя вы можете потом поэкспериментировать и изменить значение `alpha`.

    `gamma` — это **коэффициент дисконтирования**, показывающий, насколько важно учитывать будущее вознаграждение по сравнению с текущим.

    `epsilon` — это **фактор исследования/использования**, который определяет, стоит ли предпочесть исследование или использование. В нашем алгоритме в `epsilon` процентов случаев следующее действие выбирается на основе значений Q-таблицы, а в оставшихся случаях выполняется случайное действие. Это позволяет исследовать области пространства поиска, которые мы ранее не видели.

    ✅ Что касается балансировки — выбор случайного действия (исследование) действует как случайный толчок в неправильном направлении, и стержень должен научиться восстанавливаться после этих «ошибок».

### Улучшаем алгоритм

Мы также можем внести два улучшения в наш алгоритм с предыдущего урока:

- **Вычислять среднюю накопленную награду** за несколько симуляций. Мы будем выводить прогресс каждые 5000 итераций и усреднять накопленную награду за этот период. Это значит, что если мы получаем более 195 очков — мы можем считать задачу решённой с ещё более высоким качеством, чем требуется.
  
- **Вычислять максимальный средний накопленный результат**, `Qmax`, и сохранять Q-таблицу для этой лучшей модели. При запуске обучения вы заметите, что средний накопленный результат иногда начинает падать, и мы хотим сохранить значения Q-таблицы, соответствующие лучшей модели, найденной во время тренировки.

1. Собираем все накопленные награды каждой симуляции в вектор `rewards` для последующего построения графиков. (блок кода  11)

    ```python
    def probs(v,eps=1e-4):
        v = v-v.min()+eps
        v = v/v.sum()
        return v
    
    Qmax = 0
    cum_rewards = []
    rewards = []
    for epoch in range(100000):
        obs = env.reset()
        done = False
        cum_reward=0
        # == выполнить симуляцию ==
        while not done:
            s = discretize(obs)
            if random.random()<epsilon:
                # эксплуатация - выбрать действие согласно вероятностям в Q-таблице
                v = probs(np.array(qvalues(s)))
                a = random.choices(actions,weights=v)[0]
            else:
                # исследование - случайно выбрать действие
                a = np.random.randint(env.action_space.n)
    
            obs, rew, done, info = env.step(a)
            cum_reward+=rew
            ns = discretize(obs)
            Q[(s,a)] = (1 - alpha) * Q.get((s,a),0) + alpha * (rew + gamma * max(qvalues(ns)))
        cum_rewards.append(cum_reward)
        rewards.append(cum_reward)
        # == Периодически выводить результаты и вычислять среднее вознаграждение ==
        if epoch%5000==0:
            print(f"{epoch}: {np.average(cum_rewards)}, alpha={alpha}, epsilon={epsilon}")
            if np.average(cum_rewards) > Qmax:
                Qmax = np.average(cum_rewards)
                Qbest = Q.copy()
            cum_rewards=[]
    ```

Что можно заметить по этим результатам:

- **Близко к цели**. Мы очень близки к достижению цели — получить 195 накопленных очков за 100 и более последовательных запусков симуляции, и, возможно, уже достигли её! Даже если получаются меньшие числа, мы всё равно не можем точно сказать, так как усредняем по 5000 прогонов, а формальный критерий требует всего 100.
  
- **Награда начинает падать**. Иногда награда начинает снижаться, что означает, что мы можем «разрушить» уже выученные значения в Q-таблице новыми, ухудшающими ситуацию.

Это наблюдение лучше видно на графике прогресса обучения.



## График прогресса обучения

Во время обучения мы собирали значение накопленной награды на каждой итерации в вектор `rewards`. Вот как это выглядит при построении графика зависимости от номера итерации:



In [ ]:
plt.plot(rewards)

![сырое состояние прогресса](ms-assets/ml/train_progress_raw.2adfdf2daea09c59.webp)

По этой диаграмме ничего нельзя сказать, потому что из-за природы стохастического процесса обучения длина сессий сильно варьируется. Чтобы этот график имел смысл, можно вычислить **скользящее среднее** по серии экспериментальных данных, скажем, по 100. Это удобно сделать с помощью `np.convolve`: (блок кода 12)



In [ ]:
def running_average(x,window):
    return np.convolve(x,np.ones(window)/window,mode='valid')

plt.plot(running_average(rewards,100))

![прогресс обучения](ms-assets/ml/train_progress_runav.c71694a8fa9ab359.webp)



## Изменение гиперпараметров

Чтобы обучение было более стабильным, имеет смысл изменять некоторые гиперпараметры во время тренировки. В частности:

- **По коэффициенту обучения** `alpha` можно начать с значения близкого к 1, а затем постепенно снижать этот параметр. Со временем мы будем получать надежные значения вероятностей в Q-таблице, и нужно лишь небольшое корректирование, а не полное переписывание значений.

- **Увеличивать epsilon**. Можно увеличивать `epsilon` постепенно, чтобы уменьшать исследование и увеличивать использование. Вероятно, имеет смысл начать с низкого `epsilon` и подниматься до почти 1.

> **Задание 1**: Поиграйте с гиперпараметрами и попробуйте добиться большей накопленной награды. Получается ли у вас больше 195?


> **Задача 2**: Для формального решения задачи необходимо получить среднюю награду 195 за 100 последовательных запусков. Измеряйте это во время обучения и убедитесь, что вы формально решили задачу!



## Наблюдение результата в действии

Было бы интересно увидеть, как ведет себя обученная модель. Запустим симуляцию и выберем действия по той же стратегии выбора, что и во время обучения, с выборкой согласно вероятностному распределению в Q-таблице: (кодовый блок 13)



In [ ]:
obs = env.reset()
done = False
while not done:
   s = discretize(obs)
   env.render()
   v = probs(np.array(qvalues(s)))
   a = random.choices(actions,weights=v)[0]
   obs,_,done,_ = env.step(a)
env.close()

Вы должны увидеть что-то вроде этого:

![балансирующий маятник на тележке](ms-assets/ml/8-Reinforcement/2-Gym/images/cartpole-balance.gif)



## Вызов

> **Задача 3**: Здесь мы использовали финальную копию Q-таблицы, которая может быть не лучшей. Помните, что мы сохранили лучшую Q-таблицу в переменную `Qbest`! Попробуйте тот же пример, заменив `Q` на `Qbest` и посмотрите, заметите ли вы разницу.

> **Задача 4**: Здесь мы не выбирали лучшее действие на каждом шаге, а выбирали с вероятностью, соответствующей распределению. Будет ли более разумно всегда выбирать лучшее действие с наивысшим значением в Q-таблице? Это можно сделать с помощью функции `np.argmax`, чтобы определить номер действия с максимальным значением в Q-таблице. Реализуйте эту стратегию и посмотрите, улучшится ли балансирование.



## Задание

[Обучите машину в горах](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/2-Gym/assignment.md)



## Заключение

Теперь мы научились тренировать агентов добиваться хороших результатов, просто предоставляя функцию награды, которая определяет желаемое состояние игры, и давая им возможность интеллектуально исследовать пространство поиска. Мы успешно применили алгоритм Q-обучения как в дискретных, так и в непрерывных средах, но с дискретными действиями.

Важно также изучать ситуации, когда пространство действий тоже непрерывно, а пространство наблюдений намного сложнее, например изображение с экрана игры Atari. В таких задачах часто необходимо использовать более мощные методы машинного обучения, такие как нейронные сети, чтобы добиться хороших результатов. Эти более продвинутые темы будут предметом нашего следующего более продвинутого курса по ИИ.



---

## ✅ Self-check tasks

Implement the function in the 'your code' cell, then run the check cell — it runs randomized tests and shows the first mismatch. Run the helper cell below once first.

In [ ]:
# ── task-checking helper cell — run once ──
import copy
import numpy as np
import math, statistics, itertools, collections, functools, heapq, random

try:
    import sys; sys.stdout.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

_L = {"noFn":"❓ Define a function `{e}` in the solution cell and run it","runErr":"❌ Test {i}: {t}: {m}","wrong":"❌ Test {i}: {r}","badRef":"⚠️ Check harness broke (generator/reference): {t}: {m}","inL":"Input","expL":"Expected","gotL":"Got","ok":"✅ Correct! All {n} tests passed."}

def _nb_copy_kw(kw):
    return {k: (v.copy() if isinstance(v, np.ndarray) else copy.deepcopy(v)) for k, v in kw.items()}

def _nb_jsafe(x, depth=0):
    if depth > 6: return "…"
    if isinstance(x, np.ndarray): return _nb_jsafe(x.tolist(), depth + 1)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, float): return round(x, 6) if math.isfinite(x) else str(x)
    if isinstance(x, (list, tuple)): return [_nb_jsafe(v, depth + 1) for v in x]
    if isinstance(x, dict): return {str(k): _nb_jsafe(v, depth + 1) for k, v in x.items()}
    if isinstance(x, (int, str, bool)) or x is None: return x
    return repr(x)

def _nb_cmp(exp, got, path=""):
    if isinstance(exp, np.ndarray) or isinstance(got, np.ndarray):
        e, g = np.asarray(exp), np.asarray(got)
        if e.shape != g.shape:
            return f"shape {tuple(g.shape)} != expected {tuple(e.shape)}"
        try:
            if np.issubdtype(e.dtype, np.number) or np.issubdtype(g.dtype, np.number):
                ef, gf = e.astype(float), g.astype(float)
                close = np.isclose(gf, ef, atol=1e-6, rtol=1e-5, equal_nan=True)
                if not bool(close.all()):
                    i = tuple(int(v) for v in np.argwhere(~close)[0])
                    return f"{path}first mismatch at index {i}: got {gf[i]:.6g} vs expected {ef[i]:.6g}"
            elif not np.array_equal(g, e):
                return f"{path}values differ"
        except Exception as ex:
            return f"{path}cannot compare: {ex}"
        return None
    if isinstance(exp, (int, float)) and not isinstance(exp, bool) and isinstance(got, (int, float)) and not isinstance(got, bool):
        if not np.isclose(float(exp), float(got), atol=1e-6, rtol=1e-5):
            return f"{path}{got} != expected {exp}"
        return None
    if isinstance(exp, (list, tuple)) and isinstance(got, (list, tuple)):
        if len(exp) != len(got):
            return f"{path}length {len(got)} != expected {len(exp)}"
        for i, (a, b) in enumerate(zip(exp, got)):
            r = _nb_cmp(a, b, path + f"[{i}].")
            if r: return r
        return None
    if isinstance(exp, dict) and isinstance(got, dict):
        if set(exp) != set(got):
            return f"{path}keys differ"
        for k in exp:
            r = _nb_cmp(exp[k], got[k], path + f"[{k!r}].")
            if r: return r
        return None
    if exp != got:
        return f"{path}{got!r} != expected {exp!r}"
    return None

def _check_task(g, ref_src, entry, n=20, seed=0):
    """g — globals() ноутбука; ref_src — gen() + эталонное решение."""
    fn = g.get(entry)
    if not callable(fn):
        print(_L["noFn"].format(e=entry)); return
    ns = {"np": np, "math": math, "statistics": statistics, "itertools": itertools,
          "collections": collections, "functools": functools, "heapq": heapq, "random": random}
    try:
        exec(ref_src, ns)
        gen, ref = ns["gen"], ns[entry]
    except Exception as e:
        print(_L["badRef"].format(t=type(e).__name__, m=e)); return
    rng = np.random.default_rng(seed)
    for i in range(n):
        try:
            kw = gen(rng)
        except Exception as e:
            print(_L["badRef"].format(t=type(e).__name__, m=e)); return
        try:
            got = fn(**_nb_copy_kw(kw))
        except Exception as e:
            print(_L["runErr"].format(i=i + 1, t=type(e).__name__, m=e)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))); return
        exp = ref(**_nb_copy_kw(kw))
        r = _nb_cmp(exp, got)
        if r:
            print(_L["wrong"].format(i=i + 1, r=r)
                  + "\n" + _L["inL"] + ": " + repr(_nb_jsafe(kw))
                  + "\n" + _L["expL"] + ": " + repr(_nb_jsafe(exp))
                  + "\n" + _L["gotL"] + ": " + repr(_nb_jsafe(got))); return
    print(_L["ok"].format(n=n))

### Task 1 — Epsilon-greedy action choice

●○○ · ~8 min

> ε-greedy is the simplest exploration strategy: with probability ε pick a uniformly random action, otherwise take the greedy argmax_a Q(s,a). For reproducibility the randomness must come from the generator passed in as an argument — draw the coin flip with rng.random() and the exploratory action with rng.integers(k).

Implement `epsilon_greedy(q_row, eps, rng)` → int action index. With probability `eps` return a uniform random action via `rng.integers(len(q_row))`; otherwise return `argmax` of the Q-row (ties break to the lowest index). Draw `rng.random()` first, exactly once, for the exploration check.

<details><summary>Hint</summary>

if rng.random() < eps: return int(rng.integers(len(q))); else return int(np.argmax(q)).

</details>


In [ ]:
import numpy as np

def epsilon_greedy(q_row, eps, rng):
    q = np.asarray(q_row, dtype=float)
    # your code
    return 0

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    k = int(rng.integers(2, 7))\n    return {\"q_row\": rng.normal(0, 1, k),\n            \"eps\": float(rng.choice([0.0, 0.1, 0.3, 0.9, 1.0])),\n            \"rng\": rng}\n\ndef epsilon_greedy(q_row, eps, rng):\n    q = np.asarray(q_row, dtype=float)\n    if rng.random() < eps:\n        return int(rng.integers(len(q)))\n    return int(np.argmax(q))", "epsilon_greedy", 20)

### Task 2 — One Q-learning update

●●○ · ~10 min

> Q-learning bootstraps toward the TD target y = r + γ·max_{a'} Q(s',a') — or just r when the transition is terminal: Q(s,a) ← Q(s,a) + α·(y − Q(s,a)). One update touches exactly one entry of the table; all other entries stay unchanged.

Implement `q_update(Q, s, a, r, s2, alpha, gamma, done)` → updated Q-table (a new array, do not rely on in-place mutation). If `done` is True the target is `r`; otherwise `r + gamma * max(Q[s2])`. Update only entry `Q[s, a]`.

<details><summary>Hint</summary>

Copy Q, compute target = r if done else r + gamma * Q[s2].max(), then Q[s,a] += alpha * (target - Q[s,a]).

</details>


In [ ]:
import numpy as np

def q_update(Q, s, a, r, s2, alpha, gamma, done):
    Q = np.asarray(Q, dtype=float).copy()
    # your code
    return Q

In [ ]:
# self-check — run after solving the cell above
_check_task(globals(), "def gen(rng):\n    S = int(rng.integers(3, 7)); A = int(rng.integers(2, 5))\n    return {\"Q\": rng.normal(0, 1, (S, A)),\n            \"s\": int(rng.integers(S)), \"a\": int(rng.integers(A)),\n            \"r\": float(rng.normal(0, 1)), \"s2\": int(rng.integers(S)),\n            \"alpha\": float(rng.uniform(0.05, 0.5)),\n            \"gamma\": float(rng.uniform(0.8, 0.99)),\n            \"done\": bool(rng.random() < 0.25)}\n\ndef q_update(Q, s, a, r, s2, alpha, gamma, done):\n    Q = np.asarray(Q, dtype=float).copy()\n    target = r if done else r + gamma * float(np.max(Q[s2]))\n    Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])\n    return Q", "q_update", 20)

---

**Go deeper**
- [Исходный урок (Microsoft, MIT)](https://github.com/microsoft/ML-For-Beginners/blob/main/8-Reinforcement/2-Gym)
- [Учебник ML (ШАД): обучение с подкреплением](https://contest.yandex.ru/tracks/ml/environment-interaction/reinforcement-learning)